In [3]:
import numpy as np
import os
import pandas as pd

# --- PATCH FOR DEEPDISH ---
np.object = object
np.bool = bool
np.int = int
np.float = float
# --------------------------

from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure
from nilearn import datasets
from bids import BIDSLayout
import deepdish as dd


# --- 1. Setup Layout and Groups ---
data_path = '/Volumes/T9/ds001486/derivatives/fmriprep'
layout = BIDSLayout(data_path, validate=False, derivatives=False)
bold_files = layout.get(suffix='bold', extension='nii.gz', desc='preproc', return_type='file')

# Your predefined groups from your research
mld_subs = ['059', '065', '067', '069', '071', '075', '076', '077', '078', '083', '088', '095', '096', '103', '106']
td_subs = ['090', '036', '013', '008', '057', '070', '023', '024', '053', '044', '034', '060', '007', '027', '010']

# --- 2. Setup Atlas and GNN-Specific Masker ---
atlas = datasets.fetch_atlas_schaefer_2018(n_rois=200, yeo_networks=7)
masker = NiftiLabelsMasker(labels_img=atlas.maps, standardize='zscore_sample', memory=None)

# IMPORTANT: vectorize=False keeps the 400x400 shape for GNNs
# 1. Define BOTH measures (Standard and Partial)
conn_measure = ConnectivityMeasure(kind='correlation', vectorize=False, discard_diagonal=False, standardize='zscore_sample')
pconn_measure = ConnectivityMeasure(kind='partial correlation', vectorize=False, discard_diagonal=False, standardize='zscore_sample')

output_dir = os.path.expanduser('~/Desktop/math_results_braingnn')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print("Starting extraction and saving to .h5...")

for bold_path in bold_files:
    filename = os.path.basename(bold_path)
    
    # Task Filtering (Only Math tasks)
    if 'task-Mult' in filename:
        current_task = 1
    elif 'task-Sub' in filename:
        current_task = 0
    else:
        continue 

    # Confound Handling
    parts = filename.split('_')
    essential_parts = [p for p in parts if any(x in p for x in ['sub-', 'ses-', 'task-', 'run-'])]
    confound_name = "_".join(essential_parts) + "_desc-confounds_timeseries.tsv"
    confound_path = os.path.join(os.path.dirname(bold_path), confound_name)

    if not os.path.exists(confound_path):
        continue

    sub_id = filename.split('_')[0].split('-')[1] 
    group_label = 1 if sub_id in mld_subs else 0 
    
    try:
        df = pd.read_csv(confound_path, sep='\t')
        df_clean = df.select_dtypes(include=[np.number]).fillna(0).dropna(axis=1, how='all')
        
        time_series = masker.fit_transform(bold_path, confounds=df_clean)
        
        if np.any(np.isnan(time_series)): continue
            
        # 2. Extract BOTH matrices
        corr_matrix = conn_measure.fit_transform([time_series])[0]
        pcorr_matrix = pconn_measure.fit_transform([time_series])[0]
        
        # 3. Save exactly how 02-process_data.py saves it
        unique_id = f"sub-{sub_id}_task-{'Mult' if current_task==1 else 'Sub'}"
        save_path = os.path.join(output_dir, f'{unique_id}.h5')
        
        # This dictionary structure exactly matches what BrainGNN's Dataset class expects
        dd.io.save(save_path, {
            'corr': corr_matrix,
            'pcorr': pcorr_matrix,
            'label': group_label
        })
        
        print(f"Saved: {unique_id}.h5")
        
    except Exception as e:
        print(f"Error on sub-{sub_id}: {e}")

print(f"\nSUCCESS: BrainGNN-ready data saved to {output_dir}")

[fetch_atlas_schaefer_2018] Dataset found in /Users/jchong058/nilearn_data/schaefer_2018
Starting extraction and saving to .h5...


/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_15263/3797686369.py:70: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  time_series = masker.fit_transform(bold_path, confounds=df_clean)


Saved: sub-007_task-Mult.h5


/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_15263/3797686369.py:70: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  time_series = masker.fit_transform(bold_path, confounds=df_clean)


Saved: sub-007_task-Mult.h5


...

Saved: sub-106_task-Mult.h5


/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_15263/3797686369.py:70: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  time_series = masker.fit_transform(bold_path, confounds=df_clean)


Saved: sub-106_task-Sub.h5


/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_15263/3797686369.py:70: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  time_series = masker.fit_transform(bold_path, confounds=df_clean)


Saved: sub-106_task-Sub.h5

SUCCESS: BrainGNN-ready data saved to /Users/jchong058/Desktop/math_results_braingnn
